In [0]:
# makes a databricks utilities object with file path should work if you have the volume?
dbutils.fs.ls("/Volumes/workspace/default/osu_energy")

In [0]:
#this is just an example for what it looks for a single one of the meter files

path = "/Volumes/workspace/default/osu_energy/meter-readings-jan-2025.csv"

# uses a spark session to read the csv to handle large data (pandas may run out of RAM) and has optimized joins
meter_jan_df = (spark.read
      .option("header", True) #first row of column names
      .option("inferSchema", True) #spark infers data types (instead of just leaving as strings)
      .csv(path))

display(meter_jan_df)

In [0]:
#now reading all of the meter readings
base = "dbfs:/Volumes/workspace/default/osu_energy"
meter_all = (spark.read
  .option("header", True)
  .option("inferSchema", True)
  .csv(f"{base}/meter-readings-*-2025.csv"))

In [0]:
#now loading in the building metadata and the weather data
building = (spark.read
  .option("header", True)
  .option("inferSchema", True)
  .csv(f"{base}/building_metadata.csv"))

weather = (spark.read
  .option("header", True)
  .option("inferSchema", True)
  .csv(f"{base}/weather_data_hourly_2025.csv"))

display(building.limit(5))
display(weather.limit(5))

In [0]:
#after the first load apparently a cleaned parquet/delta is significantly faster to reread than csv "uses an optimized, columnar storage format (Parquet) with an intelligent metadata layer (transaction log), which enables advanced data skipping and query optimization"

(meter_all.write.mode("overwrite").format("delta")
 .save(f"{base}/_delta/meter_2025"))

(weather.write.mode("overwrite").format("delta")
 .save(f"{base}/_delta/weather_hourly_2025"))

(building.write.mode("overwrite").format("delta")
 .save(f"{base}/_delta/building_metadata"))

In [0]:
meter_all.columns

In [0]:
building.printSchema()

In [0]:
from pyspark.sql import functions as F

meter_k = meter_all.withColumn(
    "bldg_key",
    F.regexp_replace(F.col("simscode").cast("string"), r"[^0-9]", "")
)

bldg_k = building.withColumn(
    "bldg_key",
    F.regexp_replace(F.col("buildingnumber").cast("string"), r"[^0-9]", "")
)

meter_building = meter_k.join(bldg_k, "bldg_key", "left")

meter_building.select(
    F.mean(F.col("buildingnumber").isNotNull().cast("double")).alias("match_rate")
).show()

In [0]:
#lets look at what isn't matching
meter_building.filter(F.col("buildingnumber").isNull()) \
    .select("simscode", "siteid", "sitename", "utility") \
    .groupBy("simscode").count() \
    .orderBy(F.desc("count")) \
    .show(50, truncate=False)

In [0]:
meter_building.filter(F.col("buildingname").isNull()) \
    .select("simscode", "bldg_key") \
    .groupBy("simscode", "bldg_key").count() \
    .orderBy(F.desc("count")) \
    .show(50, truncate=False)

In [0]:
# let's see if we can join the remaining through names
building.select("buildingname").distinct().show(50, truncate=False)
building.select("formalname").distinct().show(50, truncate=False)
building.select("alsoknownas").distinct().show(50, truncate=False)

In [0]:
# See meters for each sim code
meter_all.groupBy("simscode").agg(F.countDistinct("meterid").alias("meters")).orderBy(F.desc("meters")).show(20)

In [0]:
def norm_str(col):
    return F.regexp_replace(
        F.regexp_replace(F.lower(F.trim(col)), r"[^a-z0-9 ]", " "),
        r"\s+",
        " "
    )

meter_name = meter_all.withColumn("name_key", norm_str(F.col("sitename")))
bldg_formal = building.withColumn("name_key", norm_str(F.col("formalname")))

join_formal = meter_name.join(bldg_formal, "name_key", "left")

join_formal.select(
    F.mean(F.col("buildingnumber").isNotNull().cast("double")).alias("match_rate")
).show()

erm it seems like name works a bit after normalizing

In [0]:
bcols = ["buildingnumber","buildingname","campusname","grossarea","latitude","longitude","formalname","address","city","state","postalcode"]

# keep only what exists + rename with fb_ prefix
name_fallback = join_formal.select(
    "meterid", "readingtime",
    *[c for c in bcols if c in join_formal.columns]
)

for c in bcols:
    if c in name_fallback.columns:
        name_fallback = name_fallback.withColumnRenamed(c, f"fb_{c}")

In [0]:
key = ["meterid", "readingtime"]

df = meter_building.join(name_fallback, on=key, how="left")

# coalesce each building column from meter_building first, then fallback
for c in bcols:
    if c in meter_building.columns and f"fb_{c}" in df.columns:
        df = df.withColumn(c, F.coalesce(F.col(c), F.col(f"fb_{c}")))

# drop the fb_ columns
df = df.drop(*[f"fb_{c}" for c in bcols if f"fb_{c}" in df.columns])

final = df

In [0]:
final.select(
    F.mean(F.col("buildingnumber").isNotNull().cast("double")).alias("final_match_rate")
).show()

Name match didn't add anything the simcode and buildingnumber already matched :/

In [0]:
#energy types
final.groupBy("utility", "readingunitsdisplay", "readingunits") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(100, truncate=False)

In [0]:
#utilities alone
final.groupBy("utility").count().orderBy(F.desc("count")).show(100, truncate=False)

In [0]:
df = final.withColumn("is_mapped_building", F.col("buildingnumber").isNotNull()) \
          .withColumn("entity_name", F.coalesce(F.col("formalname"), F.col("sitename"))) \
          .withColumn("entity_id", F.coalesce(F.col("buildingnumber"), F.concat(F.lit("UNMAPPED:"), F.col("sitename"))))

In [0]:
from pyspark.sql.window import Window

entity_totals = (df.groupBy("entity_id", "entity_name", "is_mapped_building", "utility", "readingunitsdisplay")
    .agg(
        F.sum("readingvalue").alias("total_usage"),
        F.count("*").alias("n_rows"),
        F.countDistinct("meterid").alias("n_meters")
    )
)

w = Window.partitionBy("utility", "readingunitsdisplay").orderBy(F.desc("total_usage"))

top10_entities = (entity_totals
    .withColumn("rank", F.row_number().over(w))
    .filter(F.col("rank") <= 10)
    .orderBy("utility", "readingunitsdisplay", "rank")
)

top10_entities.show(200, truncate=False)

In [0]:
top10_mapped = top10_entities.filter(F.col("is_mapped_building") == True)
top10_mapped.show(200, truncate=False)

In [0]:
top10_unmapped = top10_entities.filter(F.col("is_mapped_building") == False)
top10_unmapped.show(200, truncate=False)

In [0]:

import matplotlib.pyplot as plt

UTILITY = "ELECTRICITY"   # change: HEAT, GAS, COOLING, STEAM, ...
TOP_N = 15

# Build unified "entity" identity: mapped buildings vs unmapped sites
df = (final
      .withColumn("is_mapped_building", F.col("buildingnumber").isNotNull())
      .withColumn("entity_name", F.coalesce(F.col("formalname"), F.col("sitename")))
      .withColumn("entity_id", F.coalesce(F.col("buildingnumber"),
                                          F.concat(F.lit("UNMAPPED:"), F.col("sitename"))))
     )

filtered = df.filter(F.col("utility") == UTILITY)

entity_totals = (filtered.groupBy("entity_id", "entity_name", "is_mapped_building", "readingunitsdisplay")
                 .agg(F.sum("readingvalue").alias("total_usage"))
                )

top_mapped = (entity_totals.filter(F.col("is_mapped_building") == True)
              .orderBy(F.desc("total_usage"))
              .limit(TOP_N)
              .toPandas())

top_unmapped = (entity_totals.filter(F.col("is_mapped_building") == False)
                .orderBy(F.desc("total_usage"))
                .limit(TOP_N)
                .toPandas())

def shorten(s, n=35):
    s = "" if s is None else str(s)
    return s if len(s) <= n else s[:n-1] + "…"

# --- Chart 1: Top mapped buildings ---
if len(top_mapped) > 0:
    units = top_mapped["readingunitsdisplay"].iloc[0]
    labels = [shorten(x) for x in top_mapped["entity_name"]]
    values = top_mapped["total_usage"].tolist()

    plt.figure()
    plt.bar(labels, values)
    plt.xticks(rotation=60, ha="right")
    plt.ylabel(f"Total ({units})")
    plt.title(f"Top {TOP_N} mapped buildings by {UTILITY}")
    plt.tight_layout()
    plt.show()
else:
    print("No mapped rows found for", UTILITY)

# --- Chart 2: Top unmapped sites ---
if len(top_unmapped) > 0:
    units = top_unmapped["readingunitsdisplay"].iloc[0]
    labels = [shorten(x) for x in top_unmapped["entity_name"]]
    values = top_unmapped["total_usage"].tolist()

    plt.figure()
    plt.bar(labels, values)
    plt.xticks(rotation=60, ha="right")
    plt.ylabel(f"Total ({units})")
    plt.title(f"Top {TOP_N} unmapped sites by {UTILITY}")
    plt.tight_layout()
    plt.show()
else:
    print("No unmapped rows found for", UTILITY)

display(top_mapped)
display(top_unmapped)